In [6]:
import torch
import sys
from pathlib import Path
import torch, torch.nn as nn
import torchvision.models as models

# Add the project root (parent of NoteBooks) to Python's path
project_root = Path.cwd().parent  # if notebook is in NoteBooks/
sys.path.insert(0, str(project_root))

from src.training.train_cnn import create_cnn_model
from src.models.mlp import create_mlp_model_from_splits
from src.data.loader import *
from src.data.preprocessing import * # need thsi for just a sec. 
from scripts.ensemble_evalute import final_evaluate_mlp_on_test, final_evaluate_cnn_on_test
import numpy as np
import random 
from src.utils.model_registry import ModelRegistry # just get it all for now. 
from src.data.preprocessing import *

In [7]:
# constants. 
projections_path = "../IU-xray/indiana_projections.csv"
reports_path = "../IU-xray\indiana_reports.csv"
image_dir = '../IU-xray/images/images_normalized'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = Path("../models")
# where we can find the fetchers later. 
SPLITS_DIR = Path("../data/splits")
MODELS_DIR = Path("../models")

In [8]:
# create the data splits for training and validation purposes. Trust. 
modes = ["abnormal", "pneumonia"]
for mode in modes:
    create_train_test_split(projections_path, reports_path, mode)

  Train:      2672 (63.7% pos)
  Validation:  573 (63.7% pos)
  Test:        573 (63.7% pos)

✅ All splits saved to ..\data\splits
  Train:      2672 (7.6% pos)
  Validation:  573 (7.5% pos)
  Test:        573 (7.5% pos)

✅ All splits saved to ..\data\splits


In [9]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    # For deterministic behavior (slower but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

registry = ModelRegistry("../models/registry.json")


In [10]:
modes = ["abnormal", "pneumonia"]
seeds = [42, 123, 456, 789, 1024, 2048, 3407, 5555, 7777, 9999]

for mode in modes:
    for seed in seeds:
        # don't do this new everytime, just read the one that is already there. 
        # df = build_master_df_rich(projections_path, reports_path, mode) # get EVERYTHING. 

        # this should do the trick
        train_df = pd.read_csv(SPLITS_DIR / f"{mode}_train.csv")
        val_df = pd.read_csv(SPLITS_DIR / f"{mode}_val.csv")
        test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")  # Keep untouched

        # subset = True
        # if subset == True:
        #     train_df = train_df.sample(n=200, random_state=42)
        #     val_df = val_df.sample(n=50, random_state=42)
        #     test_df = test_df.sample(n=50, random_state=42)

        # 1. Train MLP
        print("Training MLP's")
        set_seed(seed)
        mlp_model_name = f"mlp_{mode}"
        text_amount = "medium" # use the medium amount of text data
        model_type = "single" # not ensemble, single mode. 
        # returns a state dict, not the model
        MLP_model_state_dict, errors_train, errors_test, losses_train, losses_test = create_mlp_model_from_splits(train_df, val_df, test_df, mode, text_amount, model_type)
        mlp_test_metrics = final_evaluate_mlp_on_test(MLP_model_state_dict, mode, text_amount)
        mlp_auc = mlp_test_metrics["auc"]
        mlp_should_save, mlp_reason, mlp_improvement = registry.should_save(mlp_model_name, mlp_auc)

        # 2. Train CNN
        print("training finetuned CNN's sire ")
        cnn_model_name = f"cnn_{mode}"
        current_best = registry.get_best_auc(cnn_model_name)
        print(f"Current best AUC for {cnn_model_name}: {current_best:.4f}")
        set_seed(seed)
        num_epochs = 25 # YIUPEEE
        CNN_model_state_dict, history = create_cnn_model(train_df, val_df, image_dir, device=device, mode=mode, num_epochs=num_epochs)
        cnn_test_metrics = final_evaluate_cnn_on_test(CNN_model_state_dict, mode, image_dir, device)
        cnn_auc = cnn_test_metrics["auc"]

        cnn_should_save, cnn_reason, cnn_improvement = registry.should_save(cnn_model_name, cnn_auc)


        if mlp_should_save:
            MLP_filename = f"best_mlp_{mode}_{text_amount}.pth"
            MLP_full_path = save_path / MLP_filename
            torch.save(MLP_model_state_dict, MLP_full_path)

            registry.register_model(
                mlp_model_name, 
                mlp_auc,
                MLP_full_path,
            )


        if cnn_should_save:
            CNN_filename = f"best_cnn_densenet_{mode}.pth"
            CNN_full_path = save_path / CNN_filename
            torch.save(CNN_model_state_dict, CNN_full_path)

            registry.register_model(
                cnn_model_name, 
                cnn_auc, 
                CNN_full_path,
                None
            )
    
registry.list_models()


Training MLP's
  Val:   (573, 1000)
  Test:  (573, 1000)
✅ Vectorizer saved to: ../models/vectorizer_abnormal_medium.pkl
Training MLP

FINAL MLP TEST EVALUATION - ABNORMAL
Test set: 573 samples (63.7% pos)
✅ Loaded vectorizer from: ..\models\vectorizer_abnormal_medium.pkl
Features: (573, 1000)
Model input dimension: 1000

************************************
  t=0.50 | Sensitivity: 73.4% | Specificity: 73.1% | FN: 97 | FP: 56
  t=0.30 | Sensitivity: 83.6% | Specificity: 57.2% | FN: 60 | FP: 89
  t=0.20 | Sensitivity: 90.4% | Specificity: 41.3% | FN: 35 | FP: 122
  t=0.10 | Sensitivity: 97.5% | Specificity: 21.2% | FN: 9 | FP: 164
  t=0.05 | Sensitivity: 99.5% | Specificity: 14.9% | FN: 2 | FP: 177

Test Accuracy : 73.3%
Test AUC-ROC  : 0.8350

Confusion Matrix:
  True Negatives  (correctly said normal)   : 152
  False Positives (incorrectly flagged)      : 56
  False Negatives (missed abnormal cases)   : 97  ← this is the dangerous one
  True Positives  (correctly caught)         : 268

In [11]:
import torchxrayvision as xrv
import numpy as np

def load_chexpert_model(device="cuda"):
    print("loading chexpert...")

    model = xrv.models.DenseNet(weihtts="densenet121-res224-all")

    model = model.to(device)
    model.eval() 

    return model

In [15]:
"""
test_chexpert_baseline.py
Load a CheXpert-pretrained model and evaluate on your dataset.
"""
import torch
import torch.nn as nn
import torchxrayvision as xrv
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

def load_chexpert_model(device="cuda"):
    """
    Load CheXpert-pretrained DenseNet-121.
    Trained on 224,316 chest X-rays from CheXpert dataset.
    """
    print("📦 Loading CheXpert-pretrained model...")
    
    # Load the model (trained on CheXpert + NIH + PC)
    model = xrv.models.DenseNet(weights="densenet121-res224-all")
    
    # The model expects single-channel input originally
    # But we can adapt it to work with our 1-channel images
    model = model.to(device)
    model.eval()
    
    print("✅ CheXpert model loaded!")
    print(f"   Trained on: CheXpert (224K), NIH (112K), PC (178K)")
    print(f"   Classes: {model.pathologies}")  # 14 pathologies
    
    return model

def evaluate_chexpert_on_your_data(model, test_df, image_dir, device="cuda"):
    """
    Evaluate CheXpert model on your test data.
    Note: CheXpert has 14 outputs, we need to extract the pneumonia one.
    """
    from src.data.dataset import ChestXRayDataset
    from src.data.loader import get_transforms_grayscale
    from torch.utils.data import DataLoader
    
    _, test_transforms = get_transforms_grayscale()
    test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    all_probs = []
    all_labels = []
    
    # Find which index is "Pneumonia" in CheXpert's output
    pneumonia_idx = model.pathologies.index("Pneumonia") if "Pneumonia" in model.pathologies else None
    print(f"Pneumonia index in CheXpert: {pneumonia_idx}")
    print(f"All pathologies: {model.pathologies}")
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            
            # CheXpert model expects 1-channel input
            # Our images are already 1-channel from transforms
            
            outputs = model(images)  # Shape: (batch, 14) - 14 pathologies
            
            # Extract pneumonia probability
            pneumonia_logits = outputs[:, pneumonia_idx]
            pneumonia_probs = torch.sigmoid(pneumonia_logits).cpu().numpy()
            
            all_probs.extend(pneumonia_probs)
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate AUC
    auc = roc_auc_score(all_labels, all_probs)
    
    print(f"\n{'='*60}")
    print(f"CHEXPERT BASELINE ON YOUR DATA")
    print(f"{'='*60}")
    print(f"Pneumonia AUC: {auc:.4f}")
    print(f"Your best CNN: 0.6392")
    print(f"Difference: {auc - 0.6392:+.4f}")
    
    if auc > 0.70:
        print("\n✅ CheXpert outperforms your model → Your training can improve!")
    else:
        print("\n⚠️ CheXpert also struggles → Data/label quality is the bottleneck")
    
    return auc, all_probs, all_labels


def test_chexpert_multiple_thresholds(all_probs, all_labels):
    """Test CheXpert at different thresholds."""
    from sklearn.metrics import confusion_matrix
    
    print("\n📊 CheXpert Threshold Analysis:")
    for t in [0.5, 0.3, 0.2, 0.1, 0.05]:
        preds = (np.array(all_probs) >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f"  t={t:.2f} | Sens: {sens:.1%} | Spec: {spec:.1%} | FN: {fn:3d} | FP: {fp:3d}")


# Run the test
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load your test data
    test_df = pd.read_csv("../data/splits/pneumonia_test.csv")
    
    # Load CheXpert model
    chexpert_model = load_chexpert_model(device)
    
    # Evaluate
    auc, probs, labels = evaluate_chexpert_on_your_data(
        chexpert_model, test_df, image_dir, device
    )
    
    test_chexpert_multiple_thresholds(probs, labels)

📦 Loading CheXpert-pretrained model...
✅ CheXpert model loaded!
   Trained on: CheXpert (224K), NIH (112K), PC (178K)
   Classes: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']
Pneumonia index in CheXpert: 8
All pathologies: ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia', 'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']

CHEXPERT BASELINE ON YOUR DATA
Pneumonia AUC: 0.5731
Your best CNN: 0.6392
Difference: -0.0661

⚠️ CheXpert also struggles → Data/label quality is the bottleneck

📊 CheXpert Threshold Analysis:
  t=0.50 | Sens: 100.0% | Spec: 0.0% | FN:   0 | FP: 530
  t=0.30 | Sens: 100.0% | Spec: 0.0% | FN:  

In [13]:
# code to look at models after they have been run. 

def verify_all_saved_models(modes=["abnormal", "pneumonia"], text_amount="medium"):
    """
    Load every saved model and re-run evaluations to verify they were saved correctly.
    Compares output to ensure consistency.
    """
    import torch
    import pandas as pd
    import numpy as np
    import pickle
    from pathlib import Path
    from sklearn.metrics import roc_auc_score
    
    SPLITS_DIR = Path("../data/splits")
    MODELS_DIR = Path("../models")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    results = {}
    
    print("\n" + "="*80)
    print("MODEL VERIFICATION - Loading all saved models and re-evaluating")
    print("="*80)
    
    for mode in modes:
        print(f"\n{'='*60}")
        print(f"VERIFYING MODE: {mode.upper()}")
        print(f"{'='*60}")
        
        results[mode] = {}
        
        # Load test data
        test_df = pd.read_csv(SPLITS_DIR / f"{mode}_test.csv")
        y_true = test_df['label'].values
        
        # ------------------------------------------------------------
        # 1. Verify MLP
        # ------------------------------------------------------------
        print("\n📦 Loading MLP...")
        mlp_path = MODELS_DIR / f"best_mlp_{mode}_{text_amount}.pth"
        
        if not mlp_path.exists():
            print(f"  ❌ MLP not found at: {mlp_path}")
        else:
            mlp_state_dict = torch.load(mlp_path, map_location=device)
            
            # Check what was saved
            if isinstance(mlp_state_dict, dict):
                if 'network.0.weight' in mlp_state_dict:
                    print(f"  ✅ Valid state dict with {len(mlp_state_dict)} keys")
                else:
                    print(f"  ⚠️ Dict contains unexpected keys: {list(mlp_state_dict.keys())[:5]}")
            else:
                print(f"  ❌ Not a state dict! Type: {type(mlp_state_dict)}")
            
            # Recreate model
            from src.models.mlp import MLP
            input_dim = mlp_state_dict['network.0.weight'].shape[1]
            mlp_model = MLP(input_nodes=input_dim).to(device)
            mlp_model.load_state_dict(mlp_state_dict)
            mlp_model.eval()
            
            # Load vectorizer
            vectorizer_path = MODELS_DIR / f"vectorizer_{mode}_{text_amount}.pkl"
            if vectorizer_path.exists():
                with open(vectorizer_path, 'rb') as f:
                    vectorizer = pickle.load(f)
                print(f"  ✅ Vectorizer loaded: {vectorizer_path}")
            else:
                print(f"  ❌ Vectorizer not found!")
                continue
            
            # Prepare features
            from src.data.nlp_preprocessing import build_nlp_pipeline, prepare_text_features
            nlp = build_nlp_pipeline()
            test_df_clean = prepare_text_features(test_df.copy(), nlp, mode=text_amount)
            X_test = vectorizer.transform(test_df_clean["text_clean"]).toarray()
            
            # Get predictions
            with torch.no_grad():
                X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
                logits = mlp_model(X_tensor).squeeze()
                mlp_probs = torch.sigmoid(logits).cpu().numpy()
            
            mlp_auc = roc_auc_score(y_true, mlp_probs)
            print(f"  📊 MLP Test AUC: {mlp_auc:.4f}")
            
            results[mode]['mlp'] = {
                'auc': mlp_auc,
                'probs': mlp_probs,
                'path': str(mlp_path)
            }
        
        # ------------------------------------------------------------
        # 2. Verify CNN
        # ------------------------------------------------------------
        print("\n🖼️ Loading CNN...")
        cnn_path = MODELS_DIR / f"best_cnn_densenet_{mode}.pth"
        
        if not cnn_path.exists():
            print(f"  ❌ CNN not found at: {cnn_path}")
        else:
            cnn_state_dict = torch.load(cnn_path, map_location=device)
            
            # Check what was saved
            if isinstance(cnn_state_dict, dict):
                if 'features.conv0.weight' in cnn_state_dict:
                    print(f"  ✅ Valid CNN state dict")
                elif 'train_loss' in cnn_state_dict:
                    print(f"  ⚠️ Saved history dict, not state dict!")
                    print(f"     Keys: {list(cnn_state_dict.keys())}")
                else:
                    print(f"  ⚠️ Unknown dict format")
            else:
                print(f"  ❌ Not a dict! Type: {type(cnn_state_dict)}")
            
            # Only proceed if it's a valid state dict
            if isinstance(cnn_state_dict, dict) and 'features.conv0.weight' in cnn_state_dict:
                from src.models.cnn import FinetunedDenseNet
                cnn_model = FinetunedDenseNet().to(device)
                cnn_model.load_state_dict(cnn_state_dict)
                cnn_model.eval()
                
                # Create test loader
                from src.data.dataset import ChestXRayDataset
                from src.data.loader import get_transforms_grayscale
                from torch.utils.data import DataLoader
                
                _, test_transforms = get_transforms_grayscale()
                test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)
                test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
                
                # Get predictions
                cnn_probs = []
                with torch.no_grad():
                    for images, _ in test_loader:
                        images = images.to(device)
                        logits = cnn_model(images).squeeze()
                        probs = torch.sigmoid(logits).cpu().numpy()
                        cnn_probs.extend(probs if hasattr(probs, '__len__') else [probs])
                
                cnn_probs = np.array(cnn_probs)
                cnn_auc = roc_auc_score(y_true, cnn_probs)
                print(f"  📊 CNN Test AUC: {cnn_auc:.4f}")
                
                results[mode]['cnn'] = {
                    'auc': cnn_auc,
                    'probs': cnn_probs,
                    'path': str(cnn_path)
                }
        
        # ------------------------------------------------------------
        # 3. Ensemble (if both models loaded)
        # ------------------------------------------------------------
        if 'mlp' in results[mode] and 'cnn' in results[mode]:
            print("\n🎯 Ensemble Check:")
            ensemble_probs = 0.5 * results[mode]['mlp']['probs'] + 0.5 * results[mode]['cnn']['probs']
            ensemble_auc = roc_auc_score(y_true, ensemble_probs)
            print(f"  Ensemble AUC: {ensemble_auc:.4f}")
            results[mode]['ensemble'] = {'auc': ensemble_auc, 'probs': ensemble_probs}
    
    # ------------------------------------------------------------
    # 4. Summary Table
    # ------------------------------------------------------------
    print("\n" + "="*80)
    print("VERIFICATION SUMMARY")
    print("="*80)
    print(f"{'Mode':<12} {'MLP AUC':<10} {'CNN AUC':<10} {'Ensemble AUC':<12} {'Status':<15}")
    print("-"*80)
    
    for mode in modes:
        mlp_auc = results[mode].get('mlp', {}).get('auc', None)
        cnn_auc = results[mode].get('cnn', {}).get('auc', None)
        ens_auc = results[mode].get('ensemble', {}).get('auc', None)
        
        mlp_str = f"{mlp_auc:.4f}" if mlp_auc else "N/A"
        cnn_str = f"{cnn_auc:.4f}" if cnn_auc else "N/A"
        ens_str = f"{ens_auc:.4f}" if ens_auc else "N/A"
        
        # Status
        if mlp_auc and cnn_auc:
            status = "✅ Complete"
        elif mlp_auc:
            status = "⚠️ CNN missing"
        elif cnn_auc:
            status = "⚠️ MLP missing"
        else:
            status = "❌ No models"
        
        print(f"{mode:<12} {mlp_str:<10} {cnn_str:<10} {ens_str:<12} {status:<15}")
    
    print("\n📁 Model paths checked:")
    print(f"  {MODELS_DIR.absolute()}")
    
    # List all files in models directory
    print("\n📄 Files in models directory:")
    for f in sorted(MODELS_DIR.glob("*")):
        size = f.stat().st_size / 1024
        print(f"  {f.name} ({size:.1f} KB)")
    
    return results


def compare_evaluation_outputs(results_dict):
    """
    Compare the evaluation outputs to ensure they match between training and loading.
    """
    print("\n" + "="*80)
    print("COMPARING EVALUATION OUTPUTS")
    print("="*80)
    
    for mode, data in results_dict.items():
        print(f"\n{mode.upper()}:")
        
        if 'mlp' in data and 'cnn' in data:
            mlp_auc = data['mlp']['auc']
            cnn_auc = data['cnn']['auc']
            ens_auc = data['ensemble']['auc']
            
            # Check if ensemble improves over best single model
            best_single = max(mlp_auc, cnn_auc)
            improvement = ens_auc - best_single
            
            print(f"  MLP:      {mlp_auc:.4f}")
            print(f"  CNN:      {cnn_auc:.4f}")
            print(f"  Ensemble: {ens_auc:.4f}")
            
            if improvement > 0:
                print(f"  ✅ Ensemble improves by +{improvement:.4f}")
            elif improvement > -0.01:
                print(f"  ⚠️ Ensemble similar to best single ({improvement:+.4f})")
            else:
                print(f"  ❌ Ensemble worse than best single ({improvement:+.4f})")
        else:
            print(f"  ⚠️ Missing models for full comparison")


# Run verification
results = verify_all_saved_models(modes=["abnormal", "pneumonia"])
compare_evaluation_outputs(results)


MODEL VERIFICATION - Loading all saved models and re-evaluating

VERIFYING MODE: ABNORMAL

📦 Loading MLP...
  ❌ MLP not found at: ..\models\best_mlp_abnormal_medium.pth

🖼️ Loading CNN...
  ✅ Valid CNN state dict
  📊 CNN Test AUC: 0.7506

VERIFYING MODE: PNEUMONIA

📦 Loading MLP...
  ✅ Valid state dict with 6 keys
  ✅ Vectorizer loaded: ..\models\vectorizer_pneumonia_medium.pkl
  📊 MLP Test AUC: 0.9375

🖼️ Loading CNN...
  ❌ CNN not found at: ..\models\best_cnn_densenet_pneumonia.pth

VERIFICATION SUMMARY
Mode         MLP AUC    CNN AUC    Ensemble AUC Status         
--------------------------------------------------------------------------------
abnormal     N/A        0.7506     N/A          ⚠️ MLP missing 
pneumonia    0.9375     N/A        N/A          ⚠️ CNN missing 

📁 Model paths checked:
  c:\Users\Sean Smith\Documents\GitHub\474-final-project\NoteBooks\..\models

📄 Files in models directory:
  best_cnn_densenet_abnormal.pth (27747.7 KB)
  best_mlp_pneumonia_medium.pth (1133.

In [14]:
def count_all_methods(reports_path):
    """Count positives for all three labeling methods."""
    import re
    
    df = pd.read_csv(reports_path)
    nlp = build_nlp_pipeline()
    
    old_pos = 0
    negex_pos = 0
    fixed_pos = 0
    
    for _, row in df.iterrows():
        text = f"{row.get('findings','')} {row.get('impression','')}"
        
        # Old method
        old_terms = ['pneumonia', 'consolidation', 'infiltrate', 'airspace disease', 'opacity']
        if simple_negation_check(text, old_terms):
            old_pos += 1
        
        # Negex method
        if has_pneumonia_fixed(text, nlp):
            negex_pos += 1
        
        # Fixed regex (only pneumonia + consolidation)
        if simple_negation_check_fixed(text):
            fixed_pos += 1
    
    total = len(df)
    print(f"\n{'='*60}")
    print(f"LABEL METHOD COMPARISON (n={total})")
    print(f"{'='*60}")
    print(f"Old (5 terms):    {old_pos:4d} positives ({old_pos/total:.1%})")
    print(f"Negex (token):    {negex_pos:4d} positives ({negex_pos/total:.1%})")
    print(f"Fixed (2 terms):  {fixed_pos:4d} positives ({fixed_pos/total:.1%})")
    
    # Show disagreements
    print(f"\n{'='*60}")
    print(f"DISAGREEMENTS")
    print(f"{'='*60}")
    
    # Old says yes, Fixed says no
    only_old = 0
    for _, row in df.iterrows():
        text = f"{row.get('findings','')} {row.get('impression','')}"
        old = simple_negation_check(text, old_terms)
        fixed = simple_negation_check_fixed(text)
        if old and not fixed:
            only_old += 1
            if only_old <= 5:
                print(f"\n  Old=YES, Fixed=NO:")
                print(f"  {text[:120]}...")
    
    print(f"\n  Total Old=YES, Fixed=NO: {only_old}")
    
    # Fixed says yes, Old says no
    only_fixed = 0
    for _, row in df.iterrows():
        text = f"{row.get('findings','')} {row.get('impression','')}"
        old = simple_negation_check(text, old_terms)
        fixed = simple_negation_check_fixed(text)
        if fixed and not old:
            only_fixed += 1
            if only_fixed <= 5:
                print(f"\n  Fixed=YES, Old=NO:")
                print(f"  {text[:120]}...")
    
    print(f"\n  Total Fixed=YES, Old=NO: {only_fixed}")

count_all_methods(reports_path)


LABEL METHOD COMPARISON (n=3851)
Old (5 terms):     685 positives (17.8%)
Negex (token):     236 positives (6.1%)
Fixed (2 terms):   262 positives (6.8%)

DISAGREEMENTS

  Old=YES, Fixed=NO:
  nan Heart size normal. Mediastinal silhouettes and pulmonary vascularity are within normal limits. Calcified lingular gr...

  Old=YES, Fixed=NO:
  Cardiomediastinal silhouette is within normal limits of size and appearance. The pulmonary vascularity is unremarkable. ...

  Old=YES, Fixed=NO:
  The heart is within normal limits in size. Surgical suture material projects over the right lung apex. The lungs are hyp...

  Old=YES, Fixed=NO:
  Bilateral patchy pulmonary opacities noted. Interval improvement in left base consolidative opacity. Pulmonary vascular ...

  Old=YES, Fixed=NO:
  nan Borderline heart size. Elevated left diaphragm. Clear right lung. Tracheostomy tube tip above the carina. Extensive ...

  Total Old=YES, Fixed=NO: 499

  Fixed=YES, Old=NO:
  The heart, pulmonary XXXX and medi